<a href="https://colab.research.google.com/github/unatisaini/flyrank-internship-ml/blob/main/work/notebooks/w02_ml_task_framing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-03 — Frame Your Lane as an ML Task

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/unatisaini/flyrank-internship-mi/blob/main/work/notebooks/w02_ml_task_framing.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane as an ML task (type)

*Classification, clustering, ranking, or scoring — which one, and why?*

This is a scoring task. I'm predicting a continuous "opportunity score" per page — not sorting pages into fixed categories (not classification), not grouping similar pages (not clustering), and not directly ordering search results (not ranking, though the score feeds into prioritization). The output is a number representing how much untapped CTR/engagement potential a page has, which content teams then use to decide what to fix first.

## 2. Target or proxy

*What would you predict? Where does that label come from — observed outcome or a defined rule?*

I'd predict a proxy target: engagement_gap = expected_ctr(avg_position) − ctr.
Since avg_position and ctr are both in the data, I can estimate the typical
CTR-by-position curve directly from these rows, then measure how far each
content asset falls below what's expected for its position. A large positive
gap = high opportunity (ranks well but underperforms on clicks). This proxy
comes from observed outcomes (real ctr and avg_position), not a fixed rule.

## 3. Success metric

*One metric you can defend. What number means 'good'?*

I'll use MAE (Mean Absolute Error) between predicted and actual engagement_gap
on held-out content assets, since it stays interpretable in CTR-percentage-point
terms. "Good" means MAE low enough that my top-flagged assets by predicted gap
reliably match the true top assets by actual gap.

## 4. The unit of analysis, as a real dataframe

*Load your lane's slice and show it: one row = one what?*

In [35]:
import os
for root, dirs, files in os.walk("data/raw"):
    for f in files:
        print(os.path.join(root, f))

data/raw/content_refresh_anonymized.csv


In [36]:
!pwd
!ls -la
!find . -iname "*.csv" 2>/dev/null

/content/flyrank-internship-ml/flyrank-internship-ml/flyrank-internship-ml/flyrank-internship-ml/flyrank-internship-ml
total 104
drwxr-xr-x 12 root root  4096 Aug  4 13:11 .
drwxr-xr-x 13 root root  4096 Aug  4 13:11 ..
-rw-r--r--  1 root root   654 Aug  4 13:11 AGENTS.md
-rw-r--r--  1 root root   654 Aug  4 13:11 CLAUDE.md
drwxr-xr-x  3 root root  4096 Aug  4 13:11 data
-rw-r--r--  1 root root  2763 Aug  4 13:11 DATA_USE.md
drwxr-xr-x  2 root root  4096 Aug  4 13:11 docs
drwxr-xr-x  8 root root  4096 Aug  4 13:11 .git
drwxr-xr-x  3 root root  4096 Aug  4 13:11 .github
-rw-r--r--  1 root root   993 Aug  4 13:11 .gitignore
-rw-r--r--  1 root root 10408 Aug  4 13:11 GUIDE.md
-rw-r--r--  1 root root  1289 Aug  4 13:11 LICENSE
drwxr-xr-x  2 root root  4096 Aug  4 13:11 notebooks
drwxr-xr-x  3 root root  4096 Aug  4 13:11 outputs
-rw-r--r--  1 root root  9769 Aug  4 13:11 README.md
-rw-r--r--  1 root root   107 Aug  4 13:11 requirements.txt
drwxr-xr-x  2 root root  4096 Aug  4 13:11 scripts

In [37]:
!git clone https://github.com/unatisaini/flyrank-internship-ml.git
%cd flyrank-internship-ml
!ls -la data/raw

Cloning into 'flyrank-internship-ml'...
remote: Enumerating objects: 142, done.
remote: Counting objects: 100% (142/142), done.
remote: Compressing objects: 100% (98/98), done.
remote: Total 142 (delta 53), reused 94 (delta 28), pack-reused 0 (from 0)
Receiving objects: 100% (142/142), 1.85 MiB | 10.00 MiB/s, done.
Resolving deltas: 100% (53/53), done.
/content/flyrank-internship-ml/flyrank-internship-ml/flyrank-internship-ml/flyrank-internship-ml/flyrank-internship-ml/flyrank-internship-ml
total 6580
drwxr-xr-x 2 root root    4096 Aug  4 13:14 .
drwxr-xr-x 3 root root    4096 Aug  4 13:14 ..
-rw-r--r-- 1 root root 6727670 Aug  4 13:14 content_refresh_anonymized.csv


In [38]:
import pandas as pd
df = pd.read_csv("data/raw/content_refresh_anonymized.csv")
print(df.shape)
print(df.columns.tolist())
df.head()

(30000, 44)
['content_id', 'client_id', 'search_volume', 'competition', 'competition_level', 'cpc', 'content_type', 'main_intent', 'word_count', 'char_count', 'provider_used', 'model_used', 'impressions_90d', 'clicks_90d', 'pageviews_90d', 'sessions_90d', 'users_90d', 'engaged_sessions_90d', 'ai_sessions_90d', 'scroll_events_90d', 'days_with_impressions', 'days_with_sessions', 'impressions_last_30d', 'clicks_last_30d', 'sessions_last_30d', 'impressions_prev_30d', 'clicks_prev_30d', 'sessions_prev_30d', 'content_age_days', 'age_tier', 'age_tier_order', 'days_since_last_update', 'freshness_tier', 'word_count_tier', 'char_count_tier', 'ctr', 'avg_position', 'engagement_rate', 'scroll_rate', 'ai_traffic_pct', 'impression_tier', 'position_tier', 'trend_direction', 'trend_pct']


,content_id,client_id,search_volume,competition,competition_level,cpc,content_type,main_intent,word_count,char_count,...,char_count_tier,ctr,avg_position,engagement_rate,scroll_rate,ai_traffic_pct,impression_tier,position_tier,trend_direction,trend_pct
0,content_304f48230142,client_f369cb89fc,10.0,0.67,HIGH,2.05,keyword article,transactional,3221.0,20457.0,...,15000-25000,0.76,10.6,5.88,4.55,0.0,good,striking,down,-41.4
1,content_a1fb4e703a9e,client_4e07408562,90.0,0.01,LOW,0.05,keyword article,informational,2481.0,15562.0,...,15000-25000,0.05,20.3,0.00,10.00,0.0,good,page_3_5,down,-57.7
2,content_9aa793d4d895,client_7f2253d7e2,0.0,0.00,LOW,0.00,keyword article,informational,3515.0,23643.0,...,15000-25000,0.09,36.5,0.00,28.57,0.0,good,page_3_5,down,-60.9
3,content_331d6c4de07b,client_19581e27de,10.0,0.00,LOW,0.00,keyword article,commercial,NaN,NaN,...,NaN,0.49,6.2,1.28,3.45,0.0,good,page_1,stable,-13.8
4,content_d99b7a2d90ca,client_3fdba35f04,0.0,0.00,LOW,0.00,keyword article,informational,2803.0,17469.0,...,15000-25000,0.13,44.0,0.00,24.29,0.0,good,page_3_5,down,-34.7


In [39]:
for col in df.columns:
  print(col)

content_id
client_id
search_volume
competition
competition_level
cpc
content_type
main_intent
word_count
char_count
provider_used
model_used
impressions_90d
clicks_90d
pageviews_90d
sessions_90d
users_90d
engaged_sessions_90d
ai_sessions_90d
scroll_events_90d
days_with_impressions
days_with_sessions
impressions_last_30d
clicks_last_30d
sessions_last_30d
impressions_prev_30d
clicks_prev_30d
sessions_prev_30d
content_age_days
age_tier
age_tier_order
days_since_last_update
freshness_tier
word_count_tier
char_count_tier
ctr
avg_position
engagement_rate
scroll_rate
ai_traffic_pct
impression_tier
position_tier
trend_direction
trend_pct


In [40]:
print(df.columns.tolist()[29:])

['age_tier', 'age_tier_order', 'days_since_last_update', 'freshness_tier', 'word_count_tier', 'char_count_tier', 'ctr', 'avg_position', 'engagement_rate', 'scroll_rate', 'ai_traffic_pct', 'impression_tier', 'position_tier', 'trend_direction', 'trend_pct']


In [41]:
import pandas as pd
df = pd.read_csv("data/raw/content_refresh_anonymized.csv")
df[["content_id", "impressions_last_30d", "clicks_last_30d",
    "impressions_prev_30d", "clicks_prev_30d", "content_age_days"]].head()

,content_id,impressions_last_30d,clicks_last_30d,impressions_prev_30d,clicks_prev_30d,content_age_days
0,content_304f48230142,578,2,987,13,187
1,content_a1fb4e703a9e,2501,2,5915,1,445
2,content_9aa793d4d895,2382,1,6089,3,141
3,content_331d6c4de07b,3626,22,4206,17,463
4,content_d99b7a2d90ca,4211,10,6452,2,263


One row = one content asset (content_id) — a single published page, with
its search demand (search_volume, competition), engagement metrics
(impressions/clicks/sessions across 90-day and 30-day windows), and
metadata (word_count, content_age_days). The dataframe above shows this:
each row is one piece of content's traffic and click performance history.

One row = one (page, query) search performance snapshot. Columns
include query, page URL, average position, impressions, clicks, and
CTR. The dataframe above shows this directly — each row is one page's
ranking and click performance for a single tracked query.

## 5. Why ML beats a fixed rule here

*What makes the pattern too messy for an if-statement?*

A fixed rule like "flag content with ctr under 2%" ignores that expected CTR
depends heavily on avg_position — position 1 naturally gets much higher CTR
than position 10. A flat threshold would flag almost everything ranked low,
even content performing exactly as expected, while missing real underperformers
at strong positions with unusually low CTR. Learning the position→expected-CTR
relationship from this data adapts to reality instead of guessing one number,
and can incorporate other signals (freshness_tier, word_count_tier, engagement_rate)
a static rule can't.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.